# SpaceX API Data Collection Methodology

This notebook documents the REST API data-collection method used for the Falcon 9 capstone. SpaceX API v4 is a live service, so the returned record count can change over time. The downstream capstone modelling uses the frozen IBM course CSVs in data/ to preserve the course exercise dataset.

In [ ]:
import json, time
from pathlib import Path
import requests
import pandas as pd

BASE = 'https://api.spacexdata.com/v4'
OUT = Path('outputs/api')
OUT.mkdir(parents=True, exist_ok=True)
SESSION = requests.Session()
SESSION.headers.update({'User-Agent': 'IBM-Applied-Data-Science-Capstone/1.0'})

def get_json(path):
    response = SESSION.get(f'{BASE}/{path.lstrip("/")}', timeout=30)
    response.raise_for_status()
    return response.json()

launches = get_json('launches/past')
print('Historical launch records returned by API:', len(launches))
with open(OUT/'launches_past_raw.json','w',encoding='utf-8') as f:
    json.dump(launches,f,indent=2)
display(pd.json_normalize(launches).head())

## 1. Identify Falcon 9 launches and related IDs

The launch records contain references to rocket, launchpad, payload, and core resources. The notebook fetches those resources and joins selected fields using their API identifiers.

In [ ]:
falcon9 = [x for x in launches if x.get('rocket')]
rocket_ids = sorted({x['rocket'] for x in falcon9 if x.get('rocket')})
launchpad_ids = sorted({x['launchpad'] for x in falcon9 if x.get('launchpad')})
payload_ids = sorted({pid for x in falcon9 for pid in (x.get('payloads') or [])})
core_ids = sorted({cid for x in falcon9 for cid in (x.get('cores') or []) if cid})
print('Rocket IDs:',len(rocket_ids),'Launchpad IDs:',len(launchpad_ids),'Payload IDs:',len(payload_ids),'Core IDs:',len(core_ids))

## 2. Retrieve reference resources with caching

Each unique identifier is fetched once. The short pause avoids unnecessarily aggressive requests. If a request fails, the notebook stops with the endpoint shown rather than silently replacing missing values.

In [ ]:
def fetch_many(resource, ids):
    result = {}
    for i, rid in enumerate(ids, start=1):
        result[rid] = get_json(f'{resource}/{rid}')
        if i < len(ids): time.sleep(0.05)
    return result

rockets = fetch_many('rockets', rocket_ids)
launchpads = fetch_many('launchpads', launchpad_ids)
payloads = fetch_many('payloads', payload_ids)
cores = fetch_many('cores', core_ids)
print('Retrieved rocket, launchpad, payload and core reference objects.')

## 3. Normalize selected launch fields

The table includes the core variables needed for later wrangling: flight number, UTC date, booster version, payload mass, orbit, launch site, payload and recovery details. The landing target is derived from actual core landing-success fields where available; missing/unknown records remain missing and are not force-labelled.

In [ ]:
rows = []
for launch in falcon9:
    rid = launch.get('rocket')
    lid = launch.get('launchpad')
    rocket = rockets.get(rid,{})
    pad = launchpads.get(lid,{})
    for payload_id in launch.get('payloads') or []:
        p = payloads.get(payload_id,{})
        for core_ref in launch.get('cores') or [None]:
            core_id = core_ref.get('core') if isinstance(core_ref,dict) else core_ref
            core = cores.get(core_id,{}) if core_id else {}
            rows.append({
                'FlightNumber': launch.get('flight_number'),
                'Date': launch.get('date_utc'),
                'BoosterVersion': rocket.get('name'),
                'PayloadMass': p.get('mass_kg'),
                'Orbit': p.get('orbit'),
                'LaunchSite': pad.get('name'),
                'LandingOutcome': core.get('landing_success'),
                'LandingType': core.get('landing_type'),
                'CoreSerial': core.get('serial'),
                'Reused': core.get('reuse_count',0) > 0 if core else None,
                'Latitude': pad.get('latitude'),
                'Longitude': pad.get('longitude'),
            })
api_df = pd.DataFrame(rows).drop_duplicates()
api_df.to_csv(OUT/'spacex_api_falcon9_records.csv',index=False)
print('Normalized Falcon 9 launch/core/payload rows:',len(api_df))
display(api_df.head(10))

## 4. Data quality and collection notes

- The API is live; record totals can change after the course snapshot.
- A launch can have multiple payloads and cores, so normalization can create more than one row per launch.
- A missing landing-success field is unknown, not automatically a failed landing.
- For the IBM machine-learning exercise, use the separate fixed course CSV and its documented binary Class target.

**GitHub source:** https://github.com/HLS-technologies/Data-Science-Capstone-Project-Report/blob/main/SpaceX_API_Data_Collection.ipynb